<img src="https://raw.githubusercontent.com/AllenSWDB/allenswdb.github.io/main/databook/resources/swdb_logo_new.jpg">  

# Extension 2: Try this on real data!


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">


*Building on Part 2 (the valid analysis workflows).*

Now that we have validated our analysis workflows on simulated data — where we know the ground
truth — we can apply them to real recordings such as the
[Visual Behavior Ophys dataset](https://allenswdb.github.io/physiology/ophys/visual-behavior/VB-Ophys.html).

Two things you can do here:

1. **Check the assumptions.** Our simulation assumed each cell's per-trial-group activity is
   independent Gaussian noise and that behavioral performance fluctuates as i.i.d. normal draws.
   With real data you can test whether those assumptions actually hold (e.g. autocorrelation in
   performance across trial groups, non-normal activity distributions, etc.).
2. **Run a validated pipeline.** Apply one of the *valid* workflows from Part 2 / Extension 3
   (train/test split, FDR-BH, or LOO-CV with a permutation test) to the real activity and
   performance arrays.

**TODO: Add more context on visual behavior experiments and links to the databook here.**

</div>


In [ ]:
from pathlib import Path

import numpy as np
from pynwb import NWBHDF5IO

# The valid workflows developed in the earlier notebooks are available in utils.py and can be
# applied directly to real data once it is loaded into the same array shapes we used for the
# simulations:
#   Perf : ndarray, shape (Ngroups,)          behavioral performance per trial group
#   DA   : ndarray, shape (Ngroups, Ncells)   average cell activity per trial group
from utils import train_test_PRCA, fwer_cells, fdr_bh_cells, loo_cv_PRCA, correlate_and_select


def open_attached_nwb(data_dir=Path("/data")):
    """Open the first attached NWB file or NWB Zarr store."""
    candidates = sorted(data_dir.glob("**/*.nwb"))
    if candidates:
        io = NWBHDF5IO(str(candidates[0]), mode="r", load_namespaces=True)
    else:
        candidates = sorted(data_dir.glob("**/*.nwb.zarr"))
        if not candidates:
            raise FileNotFoundError(f"No .nwb or .nwb.zarr file found under {data_dir}")
        from hdmf_zarr import NWBZarrIO
        io = NWBZarrIO(path=str(candidates[0]), mode="r")
    return io, io.read()


def find_dff(nwbfile):
    """Return dF/F data and timestamps, computing dF/F from fluorescence if necessary."""
    interfaces = [
        interface
        for module in nwbfile.processing.values()
        for interface in module.data_interfaces.values()
    ]
    for interface in interfaces:
        if "dff" in interface.name.lower() and hasattr(interface, "roi_response_series"):
            series = next(iter(interface.roi_response_series.values()))
            return np.asarray(series.data), np.asarray(series.timestamps)

    for interface in interfaces:
        if "fluorescence" in interface.name.lower() and hasattr(interface, "roi_response_series"):
            series = next(iter(interface.roi_response_series.values()))
            fluorescence = np.asarray(series.data)
            baseline = np.nanpercentile(fluorescence, 20, axis=0)
            return (fluorescence - baseline) / baseline, np.asarray(series.timestamps)

    raise ValueError(
        "This NWB has no ophys dF/F or fluorescence RoiResponseSeries. "
        "Choose an imaging session containing an ophys processing module."
    )


io, nwbfile = open_attached_nwb()
dff, dff_timestamps = find_dff(nwbfile)

# For Visual Behavior NWBs, each row of trials is a stimulus presentation. If the file has a
# separate stimulus_presentations table, use it instead.
presentations = getattr(nwbfile, "stimulus_presentations", nwbfile.trials)
trials = presentations.to_dataframe()
trial_dff = np.full((len(trials), dff.shape[1]), np.nan)
for trial_index, trial in trials.iterrows():
    in_presentation = (dff_timestamps >= trial.start_time) & (dff_timestamps < trial.stop_time)
    if in_presentation.any():
        trial_dff[trial_index] = dff[in_presentation].mean(axis=0)

# Group consecutive non-aborted trials and measure performance as the fraction correct.
valid_trials = ~trials.get("aborted", False).to_numpy(dtype=bool)
correct = (trials["hit"].to_numpy(dtype=bool) | trials["correct_reject"].to_numpy(dtype=bool))
block_size = 20
block_starts = np.arange(0, valid_trials.sum() - block_size + 1, block_size)
valid_dff = trial_dff[valid_trials]
valid_correct = correct[valid_trials]
DA_real = np.vstack([valid_dff[start : start + block_size].mean(axis=0) for start in block_starts])
Perf_real = np.array([valid_correct[start : start + block_size].mean() for start in block_starts])

io.close()

print(f"Built Perf_real {Perf_real.shape} and DA_real {DA_real.shape} from {len(valid_dff)} trials.")

# Once loaded, the validated analyses are one call each:
# selected = fdr_bh_cells(DA_real, Perf_real, q=0.05)
# split = train_test_PRCA(DA_real, Perf_real)
# PRCA_cv, _ = loo_cv_PRCA(DA_real, Perf_real)